<a href="https://colab.research.google.com/github/brianellis1997/Sarcasm_Detection/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/brianellis1997/Sarcasm_Detection.git

#Expirimentation with CNN Architecture for Sarcasm Detection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/train_bal_2.csv')

In [ ]:
train_bal

In [ ]:
y = train_bal['label']
X = train_bal.drop(columns=['label'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


In [ ]:
# X_test, X_eval, y_test, y_eval = train_test_split(X, y, test_size=0.2, random_state=3)

#Preparing train data

In [ ]:
X_text = X_train[['comment', 'parent_comment']]
X_numeric = X_train[['word_count', 'capital_letters_count', 'total_punctuation']]

In [ ]:
max_words = 10000  # Maximum number of words to keep based on word frequency
max_seq_length = 100  # Maximum length of sequences

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, concatenate, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model

tokenizer = Tokenizer(num_words=max_words, lower=False)
tokenizer.fit_on_texts(X_text['comment'] + ' ' + X_text['parent_comment'])

***Possibly tokenize differenlty for comment and parent comment****

In [ ]:
X_text_comment_seq = tokenizer.texts_to_sequences(X_text['comment'])
X_text_parent_comment_seq = tokenizer.texts_to_sequences(X_text['parent_comment'])

In [ ]:
X_text_comment_pad = pad_sequences(X_text_comment_seq, maxlen=max_seq_length)
X_text_parent_comment_pad = pad_sequences(X_text_parent_comment_seq, maxlen=max_seq_length)

In [ ]:
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

#Word2vec model trained on our text data

In [ ]:
import gensim
from gensim.models.phrases import Phraser, Phrases
from gensim.models.word2vec import Word2Vec
word2vec = gensim.models.Word2Vec.load('/content/drive/MyDrive/Sarcasm_Data/word2vec.model')

In [ ]:
# from gensim.models import Word2Vec

# word_index = {word: idx for idx, word in enumerate(word2vec.wv.index_to_key, start=1)}


In [ ]:
import numpy as np
from keras.layers import Embedding

# # Assume word2vec is your pre-trained Word2Vec model
# # Assume word_index is a dictionary mapping from word to integer index
# # Assume max_words is the size of your vocabulary, and word2vec.vector_size is the size of each word vector


# # max_words = len(word_index) + 1  # Adjust based on whether you start indexing from 0 or 1

# # Initialize the embedding matrix
# embedding_matrix = np.zeros((max_words, embedding_dim))

# # # Populate the embedding matrix
# # for word, i in word_index.items():
# #     if i < max_words:
# #         if word in word2vec.wv:
# #             # Words not found in the embedding index will be all zeros
# #             embedding_matrix[i] = word2vec.wv[word]



In [ ]:
# Initialize the embedding matrix
embedding_dim = word2vec.vector_size
embedding_matrix = np.zeros((max_words, embedding_dim))

# Populate the embedding matrix using the tokenizer's word_index
for word, i in tokenizer.word_index.items():
    if i < max_words:  # Ensure you don't exceed the max_words limit
        if word in word2vec.wv:
            embedding_matrix[i] = word2vec.wv[word]


In [ ]:
embedding_dim

#Preparing Test Data

In [ ]:
X_text_test = X_test[['comment', 'parent_comment']]
X_numeric_test = X_test[['word_count', 'capital_letters_count', 'total_punctuation']]

In [ ]:
X_text_comment_seq_test = tokenizer.texts_to_sequences(X_text_test['comment'])
X_text_parent_comment_seq_test = tokenizer.texts_to_sequences(X_text_test['parent_comment'])

In [ ]:
X_text_comment_pad_test = pad_sequences(X_text_comment_seq_test, maxlen=max_seq_length)
X_text_parent_comment_pad_test = pad_sequences(X_text_parent_comment_seq_test, maxlen=max_seq_length)

In [ ]:
scaler = StandardScaler()
X_numeric_scaled_test = scaler.fit_transform(X_numeric_test)

In [ ]:
pip install keras-tuner


In [ ]:
import numpy as np
import tensorflow as tf
import random

# Set the seed value
seed_value = 42

# 1. Set `PYTHONHASHSEED` environment variable at a fixed value
import os
os.environ['PYTHONHASHSEED'] = str(seed_value)

# 2. Set `python` built-in pseudo-random generator at a fixed value
random.seed(seed_value)

# 3. Set `numpy` pseudo-random generator at a fixed value
np.random.seed(seed_value)

# 4. Set the `tensorflow` pseudo-random generator at a fixed value
tf.random.set_seed(seed_value)


##CNN Text Covariates

In [ ]:
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, Dropout, Bidirectional, LSTM
from keras.models import Model
from keras import optimizers
from kerastuner import HyperModel

def build_model1(hp):
    comment_input = Input(shape=(max_seq_length,), name='comment_input')
    parent_comment_input = Input(shape=(max_seq_length,), name='parent_comment_input')

    embedding_layer = Embedding(input_dim=max_words,
                                output_dim=150,
                                weights=[embedding_matrix],
                                input_length=max_seq_length,
                                trainable=True)

    comment_embedding = embedding_layer(comment_input)
    parent_comment_embedding = embedding_layer(parent_comment_input)

    concatenated_embeddings = concatenate([comment_embedding, parent_comment_embedding], axis=1)

    conv1 = Conv1D(filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv1_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(concatenated_embeddings)
    conv1 = MaxPooling1D(pool_size=2)(conv1)

    conv2 = Conv1D(filters=hp.Int('conv2_filters', min_value=32, max_value=64, step=32),
                   kernel_size=hp.Choice('conv2_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(conv1)
    conv2 = MaxPooling1D(pool_size=2)(conv2)


    flattened = Flatten()(conv2)
    flattened = Dropout(0.5)(flattened)

    dense = Dense(hp.Int('dense_units', min_value=64, max_value=128, step=32), activation='relu')(flattened)
    output = Dense(1, activation='sigmoid')(dense)

    model = Model(inputs=[comment_input, parent_comment_input], outputs=output)

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

In [ ]:
# from kerastuner.tuners import RandomSearch

# tuner1 = RandomSearch(
#     build_model1,
#     objective='val_accuracy',
#     max_trials=5,
#     executions_per_trial=1,
#     directory='my_dir',
#     project_name='hparam_tuning'
# )

# tuner1.search_space_summary()

# tuner1.search(x=[X_text_comment_pad, X_text_parent_comment_pad],
#              y=y_train,
#              epochs=4,
#              batch_size=128,
#              validation_split=0.10)



# CNN_Text = tuner1.get_best_models(num_models=1)[0]

In [ ]:
# CNN_Text.save('/content/drive/MyDrive/Sarcasm_Data/CNN_Text.h5')

In [ ]:
from tensorflow.keras.models import load_model
CNN_Text = load_model('/content/drive/MyDrive/Sarcasm_Data/CNN_Text.h5')

In [ ]:
plot_model(CNN_Text)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions = CNN_Text.predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test})

prediction_result = (predictions > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)

#CNN Text and Non-Text Covariates

In [ ]:
import keras_tuner as kt
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, Dropout
from keras.models import Model
from keras.callbacks import EarlyStopping
from keras import optimizers

def build_model2(hp):
    comment_input = Input(shape=(max_seq_length,), name='comment_input')
    parent_comment_input = Input(shape=(max_seq_length,), name='parent_comment_input')
    numeric_input = Input(shape=(X_numeric_scaled.shape[1],), name='numeric_input')

    embedding_layer = Embedding(input_dim=max_words,
                                output_dim=150,
                                weights=[embedding_matrix],
                                input_length=max_seq_length,
                                trainable=True)

    comment_embedding = embedding_layer(comment_input)
    parent_comment_embedding = embedding_layer(parent_comment_input)

    concatenated_embeddings = concatenate([comment_embedding, parent_comment_embedding], axis=1)

    conv1 = Conv1D(filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv1_kernel_size', values=[3, 5, 7]),
                   padding='same',
                   activation='relu')(concatenated_embeddings)
    conv1 = MaxPooling1D(pool_size=2)(conv1)

    conv2 = Conv1D(filters=hp.Int('conv2_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv2_kernel_size', values=[3, 5, 7]),
                   padding='same',
                   activation='relu')(conv1)
    conv2 = MaxPooling1D(pool_size=2)(conv2)

    flattened = Flatten()(conv2)
    flattened = Dropout(0.5)(flattened)

    numeric_dense = Dense(hp.Int('numeric_dense_units', min_value=32, max_value=128, step=32), activation='relu')(numeric_input)
    numeric_dense = Dropout(0.5)(numeric_dense)
    numeric_dense = Dense(hp.Int('numeric_dense_units2', min_value=16, max_value=64, step=16), activation='relu')(numeric_dense)

    concatenated_final = concatenate([flattened, numeric_dense], axis=-1)

    dense1 = Dense(hp.Int('dense_units', min_value=32, max_value=128, step=32), activation='relu')(concatenated_final)
    dense1 = Dropout(0.5)(dense1)
    output = Dense(1, activation='sigmoid')(dense1)

    model2 = Model(inputs=[comment_input, parent_comment_input, numeric_input], outputs=output)
    model2.compile(optimizer=optimizers.Adam(hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model2


In [ ]:
# from kerastuner.tuners import RandomSearch

# early_stopping = EarlyStopping(monitor='val_loss', patience=5)

# tuner2 = RandomSearch(
#     build_model2,
#     objective='val_accuracy',
#     max_trials=5,
#     executions_per_trial=1,
#     directory='my_dir',
#     project_name='hparam_tuning2'
# )

# tuner2.search_space_summary()

# tuner2.search(x=[X_text_comment_pad, X_text_parent_comment_pad, X_numeric_scaled],
#              y=y_train,
#              epochs=4,
#              batch_size=128,
#              validation_split=0.10,
#              callbacks=[early_stopping])



# CNN_Text_NonText = tuner2.get_best_models(num_models=1)[0]

In [ ]:
# CNN_Text_NonText.save('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_NonText.h5')

In [ ]:
from tensorflow.keras.models import load_model
CNN_Text_NonText = load_model('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_NonText.h5')

In [ ]:
plot_model(CNN_Text_NonText)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions = CNN_Text_NonText.predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test,
                                               'numeric_input': X_numeric_scaled_test})

prediction_result = (predictions > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)

#CNN with BiLSTM Text and Non-Text Covariates

In [ ]:
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, Dropout, Bidirectional, LSTM
from keras.models import Model
from keras import optimizers
import keras_tuner as kt

def build_model3(hp):
    comment_input = Input(shape=(max_seq_length,), name='comment_input')
    parent_comment_input = Input(shape=(max_seq_length,), name='parent_comment_input')
    numeric_input = Input(shape=(X_numeric_scaled.shape[1],), name='numeric_input')

    embedding_layer = Embedding(input_dim=max_words,
                                output_dim=150,
                                weights=[embedding_matrix],
                                input_length=max_seq_length,
                                trainable=True)

    comment_embedding = embedding_layer(comment_input)
    parent_comment_embedding = embedding_layer(parent_comment_input)

    concatenated_embeddings = concatenate([comment_embedding, parent_comment_embedding], axis=1)

    conv1 = Conv1D(filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32),
                   kernel_size=5,
                   padding='same',
                   activation='relu')(concatenated_embeddings)
    conv1 = MaxPooling1D(pool_size=2)(conv1)

    conv2 = Conv1D(filters=hp.Int('conv2_filters', min_value=32, max_value=128, step=32),
                   kernel_size=3,
                   padding='same',
                   activation='relu')(conv1)
    conv2 = MaxPooling1D(pool_size=2)(conv2)

    bi_lstm = Bidirectional(LSTM(hp.Int('lstm_units', min_value=32, max_value=128, step=32)))(conv2)
    bi_lstm = Dropout(0.5)(bi_lstm)

    numeric_dense = Dense(64, activation='relu')(numeric_input)
    numeric_dense = Dropout(0.5)(numeric_dense)
    numeric_dense = Dense(32, activation='relu')(numeric_dense)

    concatenated_final = concatenate([bi_lstm, numeric_dense], axis=-1)

    dense1 = Dense(hp.Int('dense1_units', min_value=32, max_value=128, step=32), activation='relu')(concatenated_final)
    dense1 = Dropout(0.5)(dense1)
    output = Dense(1, activation='sigmoid')(dense1)

    model3 = Model(inputs=[comment_input, parent_comment_input, numeric_input], outputs=output)
    model3.compile(optimizer=optimizers.Adam(hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model3



In [ ]:
from kerastuner.tuners import RandomSearch

early_stopping3 = EarlyStopping(monitor='val_loss', patience=5)

tuner3 = RandomSearch(
    build_model3,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='my_dir',
    project_name='hparam_tuning3'
)

tuner3.search_space_summary()

tuner3.search(x=[X_text_comment_pad, X_text_parent_comment_pad, X_numeric_scaled],
             y=y_train,
             epochs=4,
             batch_size=128,
             validation_split=0.10,
             callbacks=[early_stopping3])



CNN_Text_NonText_BiLSTM = tuner3.get_best_models(num_models=1)[0]

In [ ]:
# CNN_Text_NonText_BiLSTM.save('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_NonText_BiLSTM.h5')

In [ ]:
from tensorflow.keras.models import load_model
CNN_Text_NonText_BiLSTM = load_model('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_NonText_BiLSTM.h5')

In [ ]:
plot_model(CNN_Text_NonText_BiLSTM)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions = CNN_Text_NonText_BiLSTM.predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test,
                                               'numeric_input': X_numeric_scaled_test})

prediction_result = (predictions > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)

#CNN with BiLSTM Text Only


In [ ]:
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, Dropout, Bidirectional, LSTM
from keras.models import Model
from keras import optimizers
from keras_tuner import HyperModel

def build_model4(hp):
    comment_input = Input(shape=(max_seq_length,), name='comment_input')
    parent_comment_input = Input(shape=(max_seq_length,), name='parent_comment_input')

    embedding_layer = Embedding(input_dim=max_words,
                                output_dim=150,
                                weights=[embedding_matrix],
                                input_length=max_seq_length,
                                trainable=True)

    comment_embedding = embedding_layer(comment_input)
    parent_comment_embedding = embedding_layer(parent_comment_input)

    concatenated_embeddings = concatenate([comment_embedding, parent_comment_embedding], axis=1)

    conv1 = Conv1D(filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv1_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(concatenated_embeddings)
    conv1 = MaxPooling1D(pool_size=2)(conv1)

    conv2 = Conv1D(filters=hp.Int('conv2_filters', min_value=32, max_value=64, step=32),
                   kernel_size=hp.Choice('conv2_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(conv1)
    conv2 = MaxPooling1D(pool_size=2)(conv2)

    bi_lstm = Bidirectional(LSTM(hp.Int('lstm_units', min_value=64, max_value=256, step=64)))(conv2)

    # Because we've added a Bi-LSTM layer, we no longer need to flatten the conv2 output
    bi_lstm = Dropout(0.5)(bi_lstm)

    dense = Dense(hp.Int('dense_units', min_value=64, max_value=128, step=32), activation='relu')(bi_lstm)
    output = Dense(1, activation='sigmoid')(dense)

    model4 = Model(inputs=[comment_input, parent_comment_input], outputs=output)

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model4.compile(optimizer=optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model4


In [ ]:
from kerastuner.tuners import RandomSearch

tuner4 = RandomSearch(
    build_model4,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='my_dir',
    project_name='hparam_tuning4'
)

tuner4.search_space_summary()

tuner4.search(x=[X_text_comment_pad, X_text_parent_comment_pad],
             y=y_train,
             epochs=4,
             batch_size=128,
             validation_split=0.10)



CNN_Text_BiLSTM = tuner4.get_best_models(num_models=1)[0]

In [ ]:
CNN_Text_BiLSTM.predict()

In [ ]:
CNN_Text_BiLSTM.save('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_BiLSTM.h5')

In [ ]:
from tensorflow.keras.models import load_model
CNN_Text_BiLSTM = load_model('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_BiLSTM.h5')

In [ ]:
plot_model(CNN_Text_BiLSTM)

Best Accuracy so Far

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions = CNN_Text_BiLSTM .predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test})

prediction_result = (predictions > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)

#CNN with BiLSTM Text and Non-Text - Attention Mechanism added for numeric data

In [ ]:
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, Dropout, Bidirectional, LSTM, Permute, Multiply, Lambda, RepeatVector
from keras.models import Model
from keras import optimizers, backend as K
from keras_tuner import HyperModel

def build_model5(hp):
    comment_input = Input(shape=(max_seq_length,), name='comment_input')
    parent_comment_input = Input(shape=(max_seq_length,), name='parent_comment_input')
    numeric_input = Input(shape=(X_numeric_scaled.shape[1],), name='numeric_input')

    embedding_layer = Embedding(input_dim=max_words,
                                output_dim=150,
                                weights=[embedding_matrix],
                                input_length=max_seq_length,
                                trainable=True)

    comment_embedding = embedding_layer(comment_input)
    parent_comment_embedding = embedding_layer(parent_comment_input)

    concatenated_embeddings = concatenate([comment_embedding, parent_comment_embedding], axis=1)

    conv1 = Conv1D(filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv1_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(concatenated_embeddings)
    conv1 = MaxPooling1D(pool_size=2)(conv1)
    conv1 = Dropout(hp.Float('dropout_conv1', min_value=0.0, max_value=0.5, step=0.1))(conv1)

    conv2 = Conv1D(filters=hp.Int('conv2_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv2_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(conv1)
    conv2 = MaxPooling1D(pool_size=2)(conv2)
    conv2 = Dropout(hp.Float('dropout_conv2', min_value=0.0, max_value=0.5, step=0.1))(conv2)

    bi_lstm = Bidirectional(LSTM(hp.Int('lstm_units', min_value=64, max_value=256, step=64)))(conv2)
    bi_lstm = Dropout(hp.Float('dropout_lstm', min_value=0.0, max_value=0.5, step=0.1))(bi_lstm)

    # Attention mechanism for numeric input
    attention_probs = Dense(1, activation='softmax', name='attention_vec')(numeric_input)
    attention_mul = Multiply()([bi_lstm, attention_probs])

    # Combine the attention output with the original bi-LSTM output
    combined_features = concatenate([attention_mul, bi_lstm])

    dense = Dense(hp.Int('dense_units', min_value=64, max_value=256, step=32), activation='relu')(combined_features)
    dense = Dropout(hp.Float('dropout_dense', min_value=0.0, max_value=0.5, step=0.1))(dense)

    output = Dense(1, activation='sigmoid')(dense)

    model5 = Model(inputs=[comment_input, parent_comment_input, numeric_input], outputs=output)

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model5.compile(optimizer=optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model5


In [ ]:
from kerastuner.tuners import RandomSearch

tuner5 = RandomSearch(
    build_model5,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='my_dir',
    project_name='hparam_tuning5'
)

tuner5.search_space_summary()

tuner5.search(x=[X_text_comment_pad, X_text_parent_comment_pad, X_numeric_scaled],
             y=y_train,
             epochs=4,
             batch_size=128,
             validation_split=0.10)



CNN_Text_NonText_Attention = tuner5.get_best_models(num_models=1)[0]

In [ ]:
# CNN_Text_NonText_Attention.save('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_NonText_Attention.h5')

In [ ]:
from tensorflow.keras.models import load_model
CNN_Text_NonText_Attention = load_model('/content/drive/MyDrive/Sarcasm_Data/CNN_Text_NonText_Attention.h5')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions = CNN_Text_NonText_Attention.predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test,
                              'numeric_input': X_numeric_scaled_test})

prediction_result = (predictions > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)

# Adding Attention Mechanism For Text as Well

In [ ]:
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, Dropout, Bidirectional, LSTM, Permute, Multiply, Lambda, RepeatVector, Activation
from keras.models import Model
from keras import optimizers, backend as K

def build_model6(hp):
    comment_input = Input(shape=(max_seq_length,), name='comment_input')
    parent_comment_input = Input(shape=(max_seq_length,), name='parent_comment_input')
    numeric_input = Input(shape=(X_numeric_scaled.shape[1],), name='numeric_input')

    embedding_layer = Embedding(input_dim=max_words,
                                output_dim=150,
                                weights=[embedding_matrix],
                                input_length=max_seq_length,
                                trainable=True)

    comment_embedding = embedding_layer(comment_input)
    parent_comment_embedding = embedding_layer(parent_comment_input)

    concatenated_embeddings = concatenate([comment_embedding, parent_comment_embedding], axis=1)

    conv1 = Conv1D(filters=hp.Int('conv1_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv1_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(concatenated_embeddings)
    conv1 = MaxPooling1D(pool_size=2)(conv1)
    conv1 = Dropout(hp.Float('dropout_conv1', min_value=0.0, max_value=0.5, step=0.1))(conv1)

    conv2 = Conv1D(filters=hp.Int('conv2_filters', min_value=32, max_value=128, step=32),
                   kernel_size=hp.Choice('conv2_kernel_size', values=[3, 5]),
                   padding='same',
                   activation='relu')(conv1)
    conv2 = MaxPooling1D(pool_size=2)(conv2)
    conv2 = Dropout(hp.Float('dropout_conv2', min_value=0.0, max_value=0.5, step=0.1))(conv2)

    bi_lstm = Bidirectional(LSTM(hp.Int('lstm_units', min_value=64, max_value=256, step=64), return_sequences=True))(conv2)
    bi_lstm = Dropout(hp.Float('dropout_lstm', min_value=0.0, max_value=0.5, step=0.1))(bi_lstm)

    # Attention layer for text data
    text_attention = Dense(1, activation='tanh')(bi_lstm)
    text_attention = Flatten()(text_attention)
    text_attention = Activation('softmax')(text_attention)
    text_attention = RepeatVector(hp.Int('lstm_units', min_value=64, max_value=256, step=64) * 2)(text_attention)
    text_attention = Permute([2, 1])(text_attention)
    text_attended = Multiply()([bi_lstm, text_attention])

    # Attention mechanism for numeric input
    numeric_attention_probs = Dense(1, activation='softmax', name='attention_vec')(numeric_input)
    attention_mul = Multiply()([text_attended, numeric_attention_probs])

    combined_features = concatenate([attention_mul, text_attended])

    dense = Dense(hp.Int('dense_units', min_value=64, max_value=256, step=32), activation='relu')(combined_features)
    dense = Dropout(hp.Float('dropout_dense', min_value=0.0, max_value=0.5, step=0.1))(dense)

    output = Dense(1, activation='sigmoid')(dense)

    model6 = Model(inputs=[comment_input, parent_comment_input, numeric_input], outputs=output)

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model6.compile(optimizer=optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model6


In [ ]:
from kerastuner.tuners import RandomSearch

tuner6 = RandomSearch(
    build_model6,
    objective='val_accuracy',
    max_trials=7,
    executions_per_trial=1,
    directory='my_dir',
    project_name='hparam_tuning6'
)

tuner6.search_space_summary()

tuner6.search(x=[X_text_comment_pad, X_text_parent_comment_pad, X_numeric_scaled],
             y=y_train,
             epochs=4,
             batch_size=128,
             validation_split=0.10)



CNN_Text_NonText_Attention2 = tuner5.get_best_models(num_models=1)[0]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions2 = CNN_Text_NonText_Attention2.predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test,
                              'numeric_input': X_numeric_scaled_test})

prediction_result = (predictions2 > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)

# Running Our Best Model on our saved Test Data

In [ ]:
import pandas as pd
test_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Test_Balanced.csv')

In [ ]:
y_test = test_bal['label']

In [ ]:
X_text_comment_seq_test = tokenizer.texts_to_sequences(test_bal['comment'])
X_text_parent_comment_seq_test = tokenizer.texts_to_sequences(test_bal['parent_comment'])

In [ ]:
X_text_comment_pad_test = pad_sequences(X_text_comment_seq_test, maxlen=max_seq_length)
X_text_parent_comment_pad_test = pad_sequences(X_text_parent_comment_seq_test, maxlen=max_seq_length)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from keras.models import Model

predictions = CNN_Text_BiLSTM .predict({'comment_input': X_text_comment_pad_test,
                             'parent_comment_input': X_text_parent_comment_pad_test})

prediction_result = (predictions > 0.5).astype('int32')

accuracy = accuracy_score(y_test, prediction_result)
print('Accuracy: %f' % accuracy)


precision = precision_score(y_test, prediction_result)
print('Precision: %f' % precision)


recall = recall_score(y_test, prediction_result)
print('Recall: %f' % recall)


f1 = f1_score(y_test, prediction_result)
print('F1 score: %f' % f1)


predictions = predictions.flatten()
auc = roc_auc_score(y_test, predictions)
print('ROC AUC: %f' % auc)